# 🎯 Multi-Agent Framework - Interactive Demo

This notebook provides a **step-by-step interactive demonstration** of the Multi-Agent Framework for Semantic Discovery & GraphRAG.

## 📋 What You'll See:

1. **🔧 Configuration** - All settings in one place
2. **✅ Service Check** - Verify Neo4j and API keys
3. **📁 Data Loading** - Load your PDF and DDL files
4. **🏗️ Builder Graph** - Construct the Knowledge Graph step-by-step
5. **🔎 Graph Inspection** - Query and visualize the results
6. **❓ Query Graph** - Ask questions with RAG
7. **🧹 Cleanup** - Reset when done

---

### 📚 Prerequisites

**Before running this notebook, make sure:**

1. **Neo4j is running** (Docker recommended):
   ```bash
   docker run -d --name neo4j-thesis \
     -p 7474:7474 -p 7687:7687 \\
     -e NEO4J_AUTH=neo4j/test_password \\
     neo4j:5
   ```

2. **`.env` file is configured** with your API keys:
   ```bash
   NEO4J_URI=bolt://localhost:7687
   NEO4J_USER=neo4j
   NEO4J_PASSWORD=test_password
   OPENROUTER_API_KEY=sk-or-your-key
   ```

3. **Dependencies installed**:
   ```bash
   pip install -e ".[dev]"
   ```

---

**📖 See [docs/RUNNING_SERVICES.md](../docs/RUNNING_SERVICES.md) for detailed setup instructions.**

## ═══════════════════════════════════════════════════════════════
## 📋 SECTION 1: CONFIGURATION
## ═══════════════════════════════════════════════════════════════

**Modify everything below to customize your run.**

All configuration is centralized here - no need to edit code in later cells!

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📋 CONFIGURATION - Modify everything here
# ═══════════════════════════════════════════════════════════════

from pathlib import Path
from typing import Any

# ─────────────────────────────────────────────────────────────
# 📁 Data Paths
# ─────────────────────────────────────────────────────────────
# Folder containing your source documents and DDL files
DATA_DIR = Path("../tests/fixtures")  # Change to your data folder

# PDF documents to process (business documentation)
PDF_FILES = [
    "sample_docs/business_glossary.txt",  # Can use .txt files directly
    "sample_docs/data_dictionary.txt",
]

# DDL (SQL schema) files to process
DDL_FILES = [
    "sample_ddl/simple_schema.sql",
    # "sample_ddl/complex_schema.sql",  # Add more as needed
]

# ─────────────────────────────────────────────────────────────
# 🧠 LLM Configuration
# ─────────────────────────────────────────────────────────────
# Models: https://openrouter.ai/models (free tier available)
LLM_MODEL_REASONING = "openai/gpt-oss-120b:free"  # For mapping, critic, grading
LLM_MODEL_EXTRACTION = "openai/gpt-oss-20b:free"   # For triplet extraction
LLM_MODEL_GENERATION = "openai/gpt-oss-120b:free"  # For answer generation

# Temperature settings:
# - 0.0 = Deterministic (same output for same input)
# - 0.3 = Slightly creative (for answer generation)
TEMPERATURE_EXTRACTION = 0.0
TEMPERATURE_REASONING = 0.0
TEMPERATURE_GENERATION = 0.3

# ─────────────────────────────────────────────────────────────
# 🗄️ Neo4j Configuration
# ─────────────────────────────────────────────────────────────
# Connection details (loaded from .env by default)
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "test_password"  # Change to match your setup

# ─────────────────────────────────────────────────────────────
# 🔍 Entity Resolution Settings
# ─────────────────────────────────────────────────────────────
# Blocking: number of similar candidates to consider
ER_BLOCKING_TOP_K = 10

# Similarity threshold: 0.0-1.0, higher = more strict
ER_SIMILARITY_THRESHOLD = 0.85

# ─────────────────────────────────────────────────────────────
# 📊 Retrieval Settings
# ─────────────────────────────────────────────────────────────
# Retrieval mode: "hybrid" | "vector" | "bm25"
RETRIEVAL_MODE = "hybrid"

# Number of results to retrieve from each channel
RETRIEVAL_VECTOR_TOP_K = 20
RETRIEVAL_BM25_TOP_K = 10

# Graph traversal depth (number of hops from retrieved nodes)
RETRIEVAL_GRAPH_DEPTH = 2

# Number of results after reranking
RERANKER_TOP_K = 5

# ─────────────────────────────────────────────────────────────
# ✅ Confidence Thresholds
# ─────────────────────────────────────────────────────────────
# Minimum confidence for auto-approving mappings
CONFIDENCE_THRESHOLD = 0.90

# Maximum retry attempts for self-reflection loops
MAX_REFLECTION_ATTEMPTS = 3
MAX_CYPHER_HEALING_ATTEMPTS = 3
MAX_HALLUCINATION_RETRIES = 3

# ─────────────────────────────────────────────────────────────
# 🔄 Feature Flags (Ablation Study)
# ─────────────────────────────────────────────────────────────
# Set to False to disable specific components (for testing/comparison)
ENABLE_SCHEMA_ENRICHMENT = True
ENABLE_CYPHER_HEALING = True
ENABLE_CRITIC_VALIDATION = True
ENABLE_RERANKER = True
ENABLE_HALLUCINATION_GRADER = True

# ─────────────────────────────────────────────────────────────
# 📝 Logging
# ─────────────────────────────────────────────────────────────
# Set to "DEBUG" for detailed logs, "INFO" for normal, "WARNING" for minimal
LOG_LEVEL = "INFO"

# ═══════════════════════════════════════════════════════════════
# ✓ Configuration loaded! Proceed to the next section.
# ═══════════════════════════════════════════════════════════════

print("""
✅ Configuration loaded successfully!

📁 Data directory: {}
📄 PDF files: {}
🗄️ DDL files: {}

🧠 LLM Models:
   - Reasoning:  {}
   - Extraction: {}
   - Generation: {}

🔧 Feature Flags:
   - Schema Enrichment:  {}
   - Cypher Healing:      {}
   - Critic Validation:   {}
   - Reranker:             {}
   - Hallucination Grader: {}
""".format(
    DATA_DIR,
    len(PDF_FILES),
    len(DDL_FILES),
    LLM_MODEL_REASONING,
    LLM_MODEL_EXTRACTION,
    LLM_MODEL_GENERATION,
    ENABLE_SCHEMA_ENRICHMENT,
    ENABLE_CYPHER_HEALING,
    ENABLE_CRITIC_VALIDATION,
    ENABLE_RERANKER,
    ENABLE_HALLUCINATION_GRADER,
))

## ═══════════════════════════════════════════════════════════════
## ✅ SECTION 2: SERVICE CHECK
## ═══════════════════════════════════════════════════════════════

Verify that all required services are running and credentials are configured.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📦 Imports & Setup
# ═══════════════════════════════════════════════════════════════

import os
import sys
import json
from IPython.display import display, HTML, Markdown

# Add project root to path
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

print("""
🔧 Setting up environment...
📂 Repository root: {}
""".format(repo_root))

# ═══════════════════════════════════════════════════════════════
# 🔍 Check Neo4j Connection
# ═══════════════════════════════════════════════════════════════

from src.graph.neo4j_client import Neo4jClient

print("\n🔍 Checking Neo4j connection...")

try:
    with Neo4jClient() as client:
        result = client.execute_cypher("RETURN 1 AS test")
    print("   ✅ Neo4j is reachable!")
    print("   📍 URI: {}".format(NEO4J_URI))
    neo4j_ok = True
except Exception as e:
    print("   ❌ Neo4j connection failed!")
    print("   Error: {}".format(e))
    print("\n   💡 Make sure Neo4j is running:")
    print("      docker run -d --name neo4j-thesis -p 7687:7687 -p 7474:7474 \\\n        -e NEO4J_AUTH=neo4j/test_password neo4j:5")
    neo4j_ok = False

# ═══════════════════════════════════════════════════════════════
# 🔑 Check API Keys
# ═══════════════════════════════════════════════════════════════

print("\n🔑 Checking API keys...")

# Load .env file
from dotenv import load_dotenv
load_dotenv(repo_root / ".env")

openrouter_key = os.getenv("OPENROUTER_API_KEY", "")

if openrouter_key and openrouter_key.startswith("sk-or-"):
    masked_key = openrouter_key[:7] + "..." + openrouter_key[-4:]
    print("   ✅ OpenRouter API key found: {}".format(masked_key))
    api_ok = True
else:
    print("   ❌ OpenRouter API key not found or invalid!")
    print("\n   💡 Get a free API key at: https://openrouter.ai/keys")
    print("   Then add it to your .env file:")
    print("      OPENROUTER_API_KEY=sk-or-your-key-here")
    api_ok = False

# ═══════════════════════════════════════════════════════════════
# 📊 System Status Summary
# ═══════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("📊 SYSTEM STATUS")
print("="*60)
print("Neo4j:      {}".format("✅ Connected" if neo4j_ok else "❌ Not connected"))
print("API Key:   {}".format("✅ Configured" if api_ok else "❌ Missing"))
print("="*60)

if not (neo4j_ok and api_ok):
    print("\n⚠️  Please fix the issues above before proceeding!")
else:
    print("\n✅ All systems ready! Proceed to Section 3.")

## ═══════════════════════════════════════════════════════════════
## 📁 SECTION 3: DATA LOADING
## ═══════════════════════════════════════════════════════════════

Load your PDF/Text documents and DDL files for processing.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📁 Load Documents
# ═══════════════════════════════════════════════════════════════

from src.ingestion.pdf_loader import load_pdf, chunk_documents

print("📁 Loading documents...\n")

# Resolve file paths
pdf_paths = [repo_root / DATA_DIR / f for f in PDF_FILES]
ddl_paths = [repo_root / DATA_DIR / f for f in DDL_FILES]

# Load and chunk documents
all_chunks = []
for pdf_path in pdf_paths:
    print("   📄 Loading: {}".format(pdf_path.name))
    try:
        documents = load_pdf(pdf_path)
        chunks = chunk_documents(documents)
        all_chunks.extend(chunks)
        print("      → {} documents, {} chunks".format(len(documents), len(chunks)))
    except Exception as e:
        print("      ❌ Error: {}".format(e))

print("\n✅ Loaded {} chunks from {} files\n".format(len(all_chunks), len(pdf_paths)))

# Show a preview of the first chunk
if all_chunks:
    print("📖 Preview of first chunk:")
    print("-" * 60)
    print(all_chunks[0].text[:200] + "..." if len(all_chunks[0].text) > 200 else all_chunks[0].text)
    print("-" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🗄️ Load DDL Files
# ═══════════════════════════════════════════════════════════════

from src.ingestion.ddl_parser import parse_ddl

print("🗄️  Loading DDL files...\n")

# Parse all DDL files
all_tables = []
for ddl_path in ddl_paths:
    print("   📋 Loading: {}".format(ddl_path.name))
    try:
        with open(ddl_path) as f:
            ddl_content = f.read()
        tables = parse_ddl(ddl_content)
        all_tables.extend(tables)
        print("      → {} tables found".format(len(tables)))
        for table in tables:
            print("         - {} ({} columns)".format(table.table_name, len(table.columns)))
    except Exception as e:
        print("      ❌ Error: {}".format(e))

print("\n✅ Loaded {} tables from {} DDL files\n".format(len(all_tables), len(ddl_paths)))

## ═══════════════════════════════════════════════════════════════
## 🏗️ SECTION 4: BUILDER GRAPH (Knowledge Graph Construction)
## ═══════════════════════════════════════════════════════════════

Run the Builder Graph to construct the Knowledge Graph from your documents and schemas.

**The Builder Graph performs these steps:**

1. **Extract Triplets** - Extract semantic relationships from text
2. **Entity Resolution** - Merge duplicate entities (blocking + LLM judge)
3. **Parse DDL** - Extract table schemas from SQL
4. **Enrich Schema** - Expand acronyms using LLM
5. **RAG Mapping** - Map tables to business concepts
6. **Validate** - Two-phase validation (Pydantic + Critic)
7. **Generate Cypher** - Create MERGE statements for Neo4j
8. **Heal Cypher** - Auto-fix syntax errors
9. **Build Graph** - Execute Cypher to build the Knowledge Graph

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🏗️ Run Builder Graph
# ═══════════════════════════════════════════════════════════════

import time
from src.graph.builder_graph import run_builder

print("🏗️  Starting Builder Graph...\n")
print(""""This will:
1. Extract triplets from documents
2. Resolve entities (merge duplicates)
3. Parse and enrich DDL schemas
4. Map tables to concepts
5. Generate and execute Cypher
""")

# Convert paths to strings for the builder
pdf_path_strings = [str(p) for p in pdf_paths]
ddl_path_strings = [str(p) for p in ddl_paths]

start_time = time.time()

# Run the builder graph
final_state = run_builder(
    raw_documents=pdf_path_strings,
    ddl_paths=ddl_path_strings,
    production=False,
)

elapsed_time = time.time() - start_time

# ═══════════════════════════════════════════════════════════════
# 📊 Results Summary
# ═══════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("📊 BUILDER GRAPH RESULTS")
print("="*60)
print("⏱️  Time: {:.2f} seconds".format(elapsed_time))
print("")
print("📝 Triplets extracted: {}".format(len(final_state.get("triplets", []))))
print("👤 Entities resolved: {}".format(len(final_state.get("entities", []))))
print("🗄️  Tables parsed: {}".format(len(final_state.get("tables", []))))
print("✨ Enriched tables: {}".format(len(final_state.get("enriched_tables", []))))
print("✅ Completed tables: {}".format(len(final_state.get("completed_tables", []))))
print("")

# Show completed tables
completed = final_state.get("completed_tables", [])
if completed:
    print("✅ Successfully processed tables:")
    for table in completed:
        print("   • {}".format(table))

if final_state.get("cypher_failed", False):
    print("⚠️  Some Cypher statements failed to heal.")
else:
    print("✅ All Cypher statements executed successfully!")

## ═══════════════════════════════════════════════════════════════
## 🔎 SECTION 5: GRAPH INSPECTION
## ═══════════════════════════════════════════════════════════════

Query the Knowledge Graph directly to see what was built.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🔎 Query the Knowledge Graph
# ═══════════════════════════════════════════════════════════════

from src.graph.neo4j_client import Neo4jClient

print("🔎 Inspecting Knowledge Graph...\n")

with Neo4jClient() as client:
    # Count nodes by type
    node_counts = client.execute_cypher("""
        MATCH (n)
        RETURN labels(n) AS labels, count(n) AS count
        ORDER BY count DESC
    """)
    
    print("📊 Node counts:")
    print("-" * 40)
    total_nodes = 0
    for row in node_counts:
        labels = row.get("labels", [])
        count = row.get("count", 0)
        total_nodes += count
        print("   {}: {}".format(labels, count))
    print("   Total: {}\n".format(total_nodes))
    
    # Show business concepts
    concepts = client.execute_cypher("""
        MATCH (c:BusinessConcept)
        RETURN c.name AS name, c.definition AS definition
        LIMIT 10
    """)
    
    print("💡 Business Concepts (sample):")
    print("-" * 40)
    for row in concepts:
        name = row.get("name", "")
        definition = row.get("definition", "")
        if definition:
            definition = definition[:80] + "..." if len(definition) > 80 else definition
            print("   • {}: {}".format(name, definition))
        else:
            print("   • {}".format(name))
    print()\n    
    # Show physical tables
    tables = client.execute_cypher("""
        MATCH (t:PhysicalTable)
        RETURN t.table_name AS table_name, t.schema_name AS schema_name
        ORDER BY table_name
    """)
    
    print("🗄️  Physical Tables:")
    print("-" * 40)
    for row in tables:
        print("   • {}.{}".format(row.get("schema_name", ""), row.get("table_name", "")))
    print()
    
    # Show mappings
    mappings = client.execute_cypher("""
        MATCH (c:BusinessConcept)-[r:MAPPED_TO]->(t:PhysicalTable)
        RETURN c.name AS concept, t.table_name AS table, r.mapping_confidence AS confidence
        ORDER BY r.mapping_confidence DESC
        LIMIT 10
    """)
    
    print("🔗 Concept → Table Mappings (sample):")
    print("-" * 40)
    for row in mappings:
        confidence = row.get("confidence", 0)
        print("   • {} → {} (confidence: {:.2f})".format(
            row.get("concept", ""),
            row.get("table", ""),
            confidence
        ))

## ═══════════════════════════════════════════════════════════════
## ❓ SECTION 6: QUERY GRAPH (Ask Questions)
## ═══════════════════════════════════════════════════════════════

Now that the Knowledge Graph is built, ask questions using RAG!

**The Query Graph performs these steps:**

1. **Hybrid Retrieval** - Vector search + BM25 + Graph traversal
2. **Reranking** - Cross-encoder reranks results
3. **Answer Generation** - LLM generates answer from context
4. **Hallucination Grading** - Self-RAG checks if answer is grounded

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ❓ Ask a Question
# ═══════════════════════════════════════════════════════════════

from src.generation.query_graph import run_query

# Example questions to try:
example_questions = [
    "What are the main entities in this domain?",
    "Which table stores customer data?",
    "How are orders related to customers?",
    "What columns are in the product table?",
]

# Enter your question (or use one of the examples)
user_query = "What tables are in the database and what do they store?"

print("❓ Question: {}\n".format(user_query))
print("🔍 Processing query through RAG pipeline...\n")

# Run the query
result = run_query(user_query)

# ═══════════════════════════════════════════════════════════════
# 📤 Display Results
# ═══════════════════════════════════════════════════════════════

print("="*60)
print("📤 ANSWER")
print("="*60)
print(result.get("final_answer", "No answer generated."))
print()

sources = result.get("sources", [])
if sources:
    print("="*60)
    print("📚 SOURCES")
    print("="*60)
    for i, source in enumerate(sources[:10], 1):
        print("   {}. {}".format(i, source))
    if len(sources) > 10:
        print("   ... and {} more".format(len(sources) - 10))
    print()

# Check if web search was used
if "[Source: Web Search]" in result.get("final_answer", ""):
    print("("⚠️  Note: Web search fallback was used - the answer may not be from your knowledge graph.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📊 Batch Queries (Test Multiple Questions)
# ═══════════════════════════════════════════════════════════════

# Define multiple questions to test
test_questions = [
    "What entities exist in the domain?",
    "Which table contains customer information?",
    "How are products and orders related?",
]

print("📊 Running batch queries...\n")
print("="*60)

for i, question in enumerate(test_questions, 1):
    print("\n❓ Question {}/{}: {}".format(i, len(test_questions), question))
    print("-" * 60)
    
    result = run_query(question)
    answer = result.get("final_answer", "")
    
    # Show answer (truncated if too long)
    if len(answer) > 300:
        answer = answer[:300] + "..."
    
    print("💡 Answer: {}\n".format(answer))

print("="*60)
print("\n✅ Batch queries complete!")

## ═══════════════════════════════════════════════════════════════
## 🧹 SECTION 7: CLEANUP & RESET
## ═══════════════════════════════════════════════════════════════

Clear the Knowledge Graph if you want to start fresh.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🧹 Clear Knowledge Graph (Optional)
# ═══════════════════════════════════════════════════════════════

# WARNING: This will delete all nodes from your Knowledge Graph!
# Uncomment the line below to execute:

# from src.graph.neo4j_client import Neo4jClient
# with Neo4jClient() as client:
#     client.execute_cypher("MATCH (n) DETACH DELETE n")
# print("✅ Knowledge Graph cleared!")

print("ℹ️  To clear the graph, uncomment the code above and run this cell.")

## ═══════════════════════════════════════════════════════════════
## 🎉 Summary
## ═══════════════════════════════════════════════════════════════

You've completed the interactive demo of the Multi-Agent Framework!

### 📚 What You Saw:

- **Builder Graph** - Documents + DDL → Knowledge Graph
- **Entity Resolution** - Merging duplicate entities
- **Schema Mapping** - Tables → Business Concepts
- **RAG Querying** - Questions → Answers from your Knowledge Graph

### 🔧 Next Steps:

1. **Try your own data** - Change `DATA_DIR` and file paths in Section 1
2. **Experiment with settings** - Toggle feature flags to compare results
3. **Explore the Neo4j Browser** - http://localhost:7474
4. **Read the documentation** - `docs/implementation/TASK.md`

### 📖 Resources:

- **Services Guide:** [`docs/RUNNING_SERVICES.md`](../docs/RUNNING_SERVICES.md)
- **Implementation Tasks:** [`docs/implementation/TASK.md`](../docs/implementation/TASK.md)
- **Project README:** [`README.md`](../README.md)

---

**Made with ❤️ for Semantic Discovery & GraphRAG**